# 04 — Evaluation and errors

Answer the primary question on the untouched final period and translate failures responsibly.

**Executed artifact:** reusable transformations live in `src/` and `sql/`; this notebook reads compact, versioned evidence rather than reprocessing 230 million events interactively.

In [1]:
import json
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
def report(name): return json.loads((ROOT/'reports'/name).read_text())

In [2]:
r=report('results_test.json')
pd.DataFrame([{'strategy':k.replace('_recall',''),'recall@20':x['estimate'],'ci95':x['ci95']} for k,x in r['metrics'].items() if k.endswith('_recall')])

,strategy,recall@20,ci95
0,global,0.005313,"[0.005017484881976528, 0.005621052469996282]"
1,recent,0.007725,"[0.00732192138295895, 0.008104218940842227]"
2,repeat,0.479005,"[0.4768472823669011, 0.4810623238458339]"
3,r,0.479005,"[0.4768472823669011, 0.4810623238458339]"
4,ra,0.544577,"[0.5424335144006069, 0.5465621602694444]"
5,c,0.543763,"[0.5415689789551359, 0.545770883576157]"


In [3]:
pd.DataFrame(r['differences']).T

,estimate,ci95
r_minus_recent_recall,0.47128,"[0.46920627728940756, 0.473283238249695]"
ra_minus_r_recall,0.065572,"[0.06449147521273495, 0.06666485109434507]"
c_minus_ra_recall,-0.000814,"[-0.0010558055865329014, -0.0005610637047953504]"
c_minus_repeat_recall,0.064758,"[0.06367456370866961, 0.0658513389564852]"
c_minus_recent_recall,0.536038,"[0.5338114295451445, 0.5380077741950919]"
c_minus_global_recall,0.53845,"[0.5362638470448765, 0.5404140652961328]"
r_minus_recent_mrr,0.323172,"[0.3213964031086802, 0.3248525888869604]"
ra_minus_r_mrr,0.00905,"[0.008626023916350029, 0.009445864126573566]"
c_minus_ra_mrr,0.020277,"[0.01963087005652177, 0.02099503593152253]"
c_minus_repeat_mrr,-0.049343,"[-0.05053741461744758, -0.048199326914765285]"


In [4]:
pd.Series(r['failure_decomposition'])

candidate_recall                    0.561245
retrieval_lost_target_mass          0.438755
ranking_lost_target_mass_c          0.017482
all_targets_missed_by_candidates    0.436539
all_targets_missed_by_c_top20       0.453959
dtype: float64

In [5]:
pd.DataFrame(r['subgroups']['target_popularity_group'])

,group,n,candidate_recall,c_recall,c_mrr,repeat_recall,delta_c_repeat_recall,delta_ci95
0,head,62491,0.690994,0.651073,0.426830,0.568739,0.082335,"[0.08001343860182532, 0.08453670661908662]"
1,mixed,1443,0.794752,0.787631,0.903365,0.777725,0.009906,"[0.006582641854381008, 0.013389903281207618]"
2,rare,1240,0.403226,0.402419,0.295498,0.397581,0.004839,"[0.000806451612903214, 0.00967741935483868]"
3,tail,135690,0.500451,0.493040,0.318513,0.435246,0.057794,"[0.05652463642616739, 0.059014951359717016]"


![Final comparison](../figures/04_incremental_recall.png)

![Failure decomposition](../figures/06_failure_decomposition.png)

## KEY FINDINGS

The primary estimate, paired interval and ablation quantify whether recent behavioral context adds practical held-out information. Retrieval and ranking losses are reported separately; rare/head/tail behavior identifies specific limits.

## LIMITATIONS

Offline prediction does not identify recommendation-caused behavior, user welfare, conversion, revenue or ROI. Intervals condition on the fitted histories and one test week.

## NEXT STEP

If complexity is justified, specify latency and concentration constraints and run a preregistered online A/B test with real exposure and inventory logs.